# Experiment 1 — Normalize-then-Solve (Qwen)

**Model:** `qwen/qwen3.8-27b` via Groq API  
**Benchmark:** 250 fixed obfuscated questions  
**Variants:** FOL obfuscation (50), Blood Relation obfuscation_l1 (50), Blood Relation obfuscation_l2 (50), Direction Sense obfuscation (50), Number Series obfuscation (50)

## Safety protocol
Run the notebook first with `TEST_MODE = True`. This processes **5 questions total (1 per variant)**.

Only after checking that the 5 calls, predictions, ground truth, evaluation, and CSV saving are correct should you change `TEST_MODE = False` and run the complete **250-question benchmark**.

The same fixed 250-question benchmark can be reused for Experiments 2 and 3.

In [ ]:
!pip -q install pandas openai tqdm


In [ ]:
import os
import re
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from IPython.display import display

# ------------------------------------------------------------
# IMPORTANT:
# True  = 5-question safety test
# False = full 250-question inference
# ------------------------------------------------------------
TEST_MODE = True
TEST_SAMPLES_PER_VARIANT = 1

MODEL_NAME = "qwen/qwen3.8-27b"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

TEMPERATURE = 0
MAX_TOKENS = 500

# Save progress every N completed questions in FULL mode.
CHECKPOINT_EVERY = 10

print("Model:", MODEL_NAME)
print("TEST_MODE:", TEST_MODE)


In [ ]:
GROQ_API_KEY = None

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

# If the secret is not configured, ask once without displaying the key.
if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is required.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL
)

print("Groq client ready.")


In [ ]:

from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

csv_names = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if not csv_names:
    raise ValueError("Please upload a CSV file.")

# Use the first uploaded CSV.
csv_name = csv_names[0]
df = pd.read_csv(csv_name)

print("Loaded:", csv_name)
print("Rows:", len(df))
print("Columns:", list(df.columns))

display(df.head())


In [ ]:


required_columns = [
    "dataset",
    "variant",
    "prompt",
    "ground_truth"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

expected = {
    ("FOL", "obfuscation"): 50,
    ("Blood_Relation", "obfuscation_l1"): 50,
    ("Blood_Relation", "obfuscation_l2"): 50,
    ("Direction_Sense", "obfuscation"): 50,
    ("Number_Series", "obfuscation"): 50,
}

print("Benchmark distribution:")
distribution = (
    df.groupby(["dataset", "variant"])
      .size()
      .reset_index(name="count")
)

display(distribution)

# Exact count checks
for (dataset, variant), expected_count in expected.items():
    actual_count = int(
        (
            (df["dataset"].astype(str) == dataset) &
            (df["variant"].astype(str) == variant)
        ).sum()
    )

    print(
        f"{dataset} / {variant}: "
        f"{actual_count} / {expected_count}"
    )

    if actual_count != expected_count:
        raise ValueError(
            f"Unexpected count for {dataset}/{variant}: "
            f"{actual_count}; expected {expected_count}."
        )

if len(df) != 250:
    raise ValueError(
        f"Expected 250 total rows, found {len(df)}."
    )

print("\n✅ Benchmark validation passed: 250 rows, 50 per variant.")


In [ ]:
#Normalize-then-Solve prompt

NORMALIZE_SOLVE_SYSTEM = r'''
You are solving an obfuscated logical reasoning problem.

Use this procedure:

1. NORMALIZE
   Internally rewrite the obfuscated problem into its clearest,
   simplest equivalent form.

   Preserve every:
   - fact
   - relationship
   - entity
   - condition
   - quantity
   - constraint
   - ordering
   - direction
   - logical polarity

   Do not add information that is not present.

2. SOLVE
   Solve the normalized problem carefully.

3. FINAL ANSWER
   Return ONLY the final answer requested by the question.

Do NOT output:
- your normalization
- your reasoning
- explanations
- labels
- extra commentary

Return only the final answer.
'''

print("Normalize-then-Solve prompt ready.")


In [ ]:
#Prediction and evaluation helpers

def normalize_answer(value):
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()

    # Remove common answer prefixes.
    prefixes = [
        "Final Answer:",
        "Final answer:",
        "Answer:",
        "answer:",
        "The answer is:",
        "the answer is:",
    ]

    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
            break

    # Normalize whitespace and letter case.
    text = re.sub(r"\s+", " ", text).strip().lower()

    # Remove one layer of simple surrounding punctuation/quotes.
    text = text.strip(" 	\n\r\"'.,;:!?")

    return text


def run_model(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": NORMALIZE_SOLVE_SYSTEM
            },
            {
                "role": "user",
                "content": str(prompt)
            }
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    content = response.choices[0].message.content

    if content is None:
        return ""

    return str(content).strip()


print("Model and evaluation helpers ready.")


In [ ]:
#Select test set or full set

if TEST_MODE:
    # Exactly one question from each of the five variants.
    parts = []

    variant_order = [
        ("FOL", "obfuscation"),
        ("Blood_Relation", "obfuscation_l1"),
        ("Blood_Relation", "obfuscation_l2"),
        ("Direction_Sense", "obfuscation"),
        ("Number_Series", "obfuscation"),
    ]

    for dataset_name, variant_name in variant_order:
        subset = df[
            (df["dataset"].astype(str) == dataset_name) &
            (df["variant"].astype(str) == variant_name)
        ]

        if len(subset) < TEST_SAMPLES_PER_VARIANT:
            raise ValueError(
                f"Not enough rows for {dataset_name}/{variant_name}."
            )

        parts.append(
            subset.head(TEST_SAMPLES_PER_VARIANT)
        )

    run_df = pd.concat(parts, ignore_index=True)

    print("TEST MODE")
    print("Rows to process:", len(run_df))

else:
    # Full benchmark: all 250 fixed questions.
    run_df = df.copy().reset_index(drop=True)

    print("FULL MODE")
    print("Rows to process:", len(run_df))

    if len(run_df) != 250:
        raise ValueError(
            f"Full benchmark should contain 250 rows, found {len(run_df)}."
        )

display(
    run_df[
        ["dataset", "variant", "row_number", "response_id", "ground_truth"]
    ]
)


In [ ]:
#Run inference with checkpointing

results = []

if TEST_MODE:
    checkpoint_file = None
else:
    checkpoint_file = (
        "experiment_01_normalize_then_solve_Qwen_250_checkpoint.csv"
    )

    # Resume safely if a prior full-run checkpoint exists.
    if os.path.exists(checkpoint_file):
        previous = pd.read_csv(checkpoint_file)
        results = previous.to_dict("records")

        processed_ids = set(
            previous["response_id"]
            .astype(str)
            .tolist()
        )

        print(
            f"Found checkpoint with {len(previous)} completed rows."
        )
    else:
        processed_ids = set()
        print("No previous full-run checkpoint found. Starting from row 1.")

print("\nStarting inference...")

for i, (_, row) in enumerate(
    tqdm(
        run_df.iterrows(),
        total=len(run_df),
        desc="Qwen Normalize-then-Solve"
    ),
    start=1
):
    response_id = str(row.get("response_id", ""))

    # Skip rows already safely stored in a full-run checkpoint.
    if not TEST_MODE and response_id in processed_ids:
        continue

    prediction = ""
    error = ""

    # Basic retry for transient API failures.
    max_attempts = 3

    for attempt in range(1, max_attempts + 1):
        try:
            prediction = run_model(row["prompt"])
            break

        except Exception as e:
            error = repr(e)

            if attempt < max_attempts:
                wait_seconds = 2 ** (attempt - 1)
                print(
                    f"\nAPI error on {response_id} "
                    f"(attempt {attempt}/{max_attempts}): {e}"
                )
                print(f"Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

    ground_truth = normalize_answer(row["ground_truth"])
    prediction_clean = normalize_answer(prediction)

    correct = (
        bool(ground_truth)
        and bool(prediction_clean)
        and prediction_clean == ground_truth
    )

    results.append({
        "dataset": row["dataset"],
        "variant": row["variant"],
        "setting": row.get("setting", ""),
        "row_number": row.get("row_number", ""),
        "response_id": response_id,
        "ground_truth": ground_truth,
        "prediction": prediction_clean,
        "correct": correct,
        "error": error,
    })

    # Save progress only in the full run.
    if not TEST_MODE and len(results) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(
            checkpoint_file,
            index=False
        )
        print(
            f"\nCheckpoint saved: "
            f"{len(results)}/250"
        )

    time.sleep(0.2)

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("INFERENCE COMPLETE")
print("=" * 60)
print("Rows in results:", len(results_df))

successful = int(
    results_df["prediction"]
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

errors = int(
    results_df["error"]
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("Successful predictions:", successful)
print("Errors:", errors)


In [ ]:
#Inspect results

display(
    results_df[
        [
            "dataset",
            "variant",
            "setting",
            "row_number",
            "ground_truth",
            "prediction",
            "correct",
            "error"
        ]
    ]
)


In [ ]:
#Accuracy by Task + Variant

valid = results_df[
    results_df["ground_truth"].astype(str).str.strip().ne("") &
    results_df["prediction"].astype(str).str.strip().ne("")
].copy()

if len(valid) == 0:
    raise ValueError(
        "No valid prediction/ground-truth pairs are available."
    )

overall_accuracy = valid["correct"].mean() * 100

print(f"Overall accuracy: {overall_accuracy:.2f}%")

variant_summary = (
    valid
    .groupby(["dataset", "variant"])["correct"]
    .agg(
        total="count",
        correct="sum",
        accuracy="mean"
    )
    .reset_index()
)

variant_summary["accuracy_percent"] = (
    variant_summary["accuracy"] * 100
)

variant_summary = variant_summary[
    ["dataset", "variant", "total", "correct", "accuracy_percent"]
]

print("\nTask + Variant results:")
display(variant_summary)


In [ ]:
#Optional setting breakdown

if "setting" in valid.columns:

    setting_summary = (
        valid
        .groupby(["dataset", "variant", "setting"])["correct"]
        .agg(
            total="count",
            correct="sum",
            accuracy="mean"
        )
        .reset_index()
    )

    setting_summary["accuracy_percent"] = (
        setting_summary["accuracy"] * 100
    )

    setting_summary = setting_summary[
        [
            "dataset",
            "variant",
            "setting",
            "total",
            "correct",
            "accuracy_percent"
        ]
    ]

    print("Task + Variant + Setting results:")
    display(setting_summary)


In [ ]:
#Save final results and summary CSVs

if TEST_MODE:
    results_file = (
        "experiment_01_normalize_then_solve_Qwen_TEST_results.csv"
    )
    summary_file = (
        "experiment_01_normalize_then_solve_Qwen_TEST_summary.csv"
    )
else:
    results_file = (
        "experiment_01_normalize_then_solve_Qwen_250_results.csv"
    )
    summary_file = (
        "experiment_01_normalize_then_solve_Qwen_250_summary.csv"
    )

results_df.to_csv(results_file, index=False)
variant_summary.to_csv(summary_file, index=False)

print("Saved:", results_file)
print("Saved:", summary_file)

if not TEST_MODE:
    print("Checkpoint file:", checkpoint_file)


In [ ]:
#Download the results to your laptop

from google.colab import files

files.download(results_file)

time.sleep(2)

files.download(summary_file)

print("\n✅ Download started.")
